<a href="https://colab.research.google.com/github/bahmedx/730/blob/main/RF_Stroke_Probability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Building First AI Model in Python**

**Import Libraries**

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

**Upload Dataset**

In [ ]:
from google.colab import files

uploaded = files.upload()

**Load and Explore Dataset**

In [ ]:
df = pd.read_csv("healthcare-dataset-stroke-data.csv")

print("Dataset Shape:", df.shape)

display(df.sample(10, random_state=42))

print("\nMissing Values:")
print(df.isnull().sum())

Handle Missing Values

In [ ]:
df["bmi"] = df["bmi"].fillna(df["bmi"].median())

print("Missing Values After Cleaning:")
print(df.isnull().sum())

**Encode Categorical Variables**

In [ ]:
categorical_columns = [
    "gender",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status"
]

encoders = {}

for col in categorical_columns:

    encoder = LabelEncoder()

    df[col] = encoder.fit_transform(df[col])

    encoders[col] = encoder

print("Encoding complete.")

display(df.head())

In [ ]:
mapping_rows = []

for col in categorical_columns:

    for value, encoded in zip(
        encoders[col].classes_,
        encoders[col].transform(encoders[col].classes_)
    ):

        mapping_rows.append({
            "Column": col,
            "Encoded Value": encoded,
            "Original Value": value
        })

mapping_df = pd.DataFrame(mapping_rows)

display(mapping_df)

**Define Features and Target**

In [ ]:
X = df.drop(columns=["id", "stroke"])
y = df["stroke"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

**Split Training and Testing Data**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

**Train Logistic Regression Model**

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

print("Random Forest Model Trained Successfully")

**Make Predictions**

In [ ]:
y_pred = model.predict(X_test)

results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

display(results.sample(10, random_state=42))

**Evaluate Performance**

Accuracy

In [ ]:
# Training Accuracy
train_pred = model.predict(X_train)
train_accuracy = accuracy_score(y_train, train_pred)

# Testing Accuracy
test_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_pred)

print(f"Training Accuracy: {train_accuracy:.2%}")
print(f"Testing Accuracy: {test_accuracy:.2%}")

Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
print(confusion_matrix(y_test, y_pred))

**Classification Report**

In [ ]:
print(classification_report(y_test, y_pred))

**Feature Importance**

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

display(feature_importance)

plt.figure(figsize=(5,3))

sns.barplot(
    data=feature_importance,
    x="Importance",
    y="Feature",
    hue="Feature",
    legend=False
)

plt.title("Feature Importance")
plt.show()

**Encode and Compare Predictions for New Patients**

In [ ]:
# Patient 1
patient_1 = pd.DataFrame({
    "gender": [1],
    "age": [55],
    "hypertension": [0],
    "heart_disease": [0],
    "ever_married": [1],
    "work_type": [3],
    "Residence_type": [1],
    "avg_glucose_level": [106],
    "bmi": [21.9],
    "smoking_status": [1]
})

# Patient 2
patient_2 = pd.DataFrame({
    "gender": [1],
    "age": [74],
    "hypertension": [0],
    "heart_disease": [0],
    "ever_married": [1],
    "work_type": [0],
    "Residence_type": [1],
    "avg_glucose_level": [110],
    "bmi": [35.5],
    "smoking_status": [2]
})

# Display predictions
for name, patient in [
    ("Patient 1 (Female)", patient_1),
    ("Patient 2 (Male)", patient_2)
]:

    prediction = model.predict(patient)[0]
    probabilities = model.predict_proba(patient)[0]

    print(f"\n{name}")
    print("-" * 40)

    print(
        "Prediction:",
        "Stroke Risk" if prediction == 1 else "No Stroke Risk"
    )

    print(f"Probability of No Stroke: {probabilities[0]:.2%}")
    print(f"Probability of Stroke: {probabilities[1]:.2%}")

**Show Patient Details and Results Together**

In [ ]:
# Group the active patient DataFrames
patients_dict = {
    "Patient 1 (Female)": patient_1,
    "Patient 2 (Male)": patient_2
}

comparison_results = []

for name, patient in patients_dict.items():
    prediction = model.predict(patient)[0]
    probabilities = model.predict_proba(patient)[0]

    comparison_results.append({
        "Patient": name,
        "Age": patient["age"].iloc[0],
        "BMI": patient["bmi"].iloc[0],
        "Glucose": patient["avg_glucose_level"].iloc[0],
        "Prediction": (
            "Stroke Risk"
            if prediction == 1
            else "No Stroke Risk"
        ),
        "Stroke Probability": f"{probabilities[1]:.2%}"
    })

comparison_df = pd.DataFrame(comparison_results)
display(comparison_df)